
# Teacher Regularization v1.0

`baseline v2.1`를 기준 backbone으로 사용하고, `DANN v1.0`의 domain adaptation loss를 참고해 teacher regularization을 추가한 notebook입니다.

구성:
- student: baseline v2.1의 `MultiViewBidirectionalCrossAttention`
- teacher target group 1: `physics`
- teacher target group 2: `image_structure`
- auxiliary: `domain_head` with GRL



In [1]:

from __future__ import annotations

import os
import sys
import json
import random
import shutil
from dataclasses import dataclass
from itertools import cycle
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.preprocessing import StandardScaler
import timm

SRC_DIR = (Path.cwd() / '../../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from augmentations import build_default_transforms
from output_paths import allocate_output_paths
from reproducibility import make_generator, seed_everything, seed_worker
from models import (
    EMAConfig,
    ModelEMA,
    MultiViewBidirectionalCrossAttentionConfig,
)

DATA_DIR = (Path.cwd() / '../../data').resolve()
FEATURE_CSV = (Path.cwd() / '../../outputs/physics_feature_analysis_v2/class_analysis_features.csv').resolve()
TRAIN_STRUCTURE_CSV = (Path.cwd() / '../../data/train_structure.csv').resolve()
DEV_STRUCTURE_CSV = (Path.cwd() / '../../data/dev_structure.csv').resolve()
assert DATA_DIR.exists(), f'data dir not found: {DATA_DIR}'
assert FEATURE_CSV.exists(), f'feature csv not found: {FEATURE_CSV}'
assert TRAIN_STRUCTURE_CSV.exists(), f'train structure csv not found: {TRAIN_STRUCTURE_CSV}'
assert DEV_STRUCTURE_CSV.exists(), f'dev structure csv not found: {DEV_STRUCTURE_CSV}'
print('DATA_DIR:', DATA_DIR)
print('FEATURE_CSV:', FEATURE_CSV)
print('TRAIN_STRUCTURE_CSV:', TRAIN_STRUCTURE_CSV)
print('DEV_STRUCTURE_CSV:', DEV_STRUCTURE_CSV)

PHYSICS_FEATURES = [
    'top_fill_ratio',
    'top_centroid_dx',
    'front_centroid_dx',
    'front_tilt',
    'top_support_width_frac',
]
IMAGE_STRUCTURE_FEATURES = [
    'abs_delta_structure_center_x',
    'top_structure_center_x',
    'front_structure_center_x',
    'mean_structure_center_x',
    'mean_structure_bbox_w',
]
ALL_TEACHER_FEATURES = PHYSICS_FEATURES + IMAGE_STRUCTURE_FEATURES
STRUCTURE_INDEX_COL = 'structure_index'
STRUCTURE_CLASS_COL = 'structure_class'

CFG = {
    'IMG_SIZE': 224,
    'EPOCHS': 100,
    'LEARNING_RATE': 3e-4,
    'BATCH_SIZE': 16,
    'SEED': 42,
    'NUM_WORKERS': 8,
    'WEIGHT_DECAY': 1e-4,
    'MIN_LR': 1e-6,
    'EMA_DECAY': 0.999,
    'EMA_USE_FOR_EVAL': True,
    'EARLY_STOPPING_PATIENCE': 5,
    'MIXUP_ALPHA': 0.1,
    'MIXUP_PROB': 0.0,
    'VIDEO_AUG_ENABLE': True,
    'VIDEO_AUG_CACHE': True,
    'UNSTABLE_START_MIN_SEC': 0.5,
    'UNSTABLE_START_MAX_SEC': 1.0,
    'UNSTABLE_FRAMES_MIN': 2,
    'UNSTABLE_FRAMES_MAX': 3,
    'STABLE_END_MIN_SEC': 9.0,
    'STABLE_END_MAX_SEC': 10.0,
    'STABLE_FRAMES_PER_VIDEO': 2,
    'BACKBONE_NAME': 'convnext_small.fb_in22k_ft_in1k',
    'ATTN_DIM': 256,
    'NUM_HEADS': 8,
    'NUM_LAYERS': 2,
    'POS_GRID': 7,
    'DROPOUT': 0.1,
    'CLASSIFIER_HIDDEN_DIM': 512,
    'CLASSIFIER_MID_DIM': 128,
    'CLASSIFIER_DROPOUT': 0.3,
    'DOMAIN_HIDDEN_DIM': 256,
    'DOMAIN_DROPOUT': 0.2,
    'PHYSICS_LOSS_WEIGHT': 0.15,
    'IMAGE_LOSS_WEIGHT': 0.15,
    'DOMAIN_LOSS_WEIGHT': 0.2,
    'STRUCTURE_LOSS_WEIGHT': 0.10,
    'GRL_WARMUP_EPOCHS': 5,
    'GRL_MAX_LAMBDA': 0.2,
    'GRL_GAMMA': 4.0,
}

seed_everything(CFG['SEED'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)



DATA_DIR: /media/hdd0/whyz/structure-stability/data
FEATURE_CSV: /media/hdd0/whyz/structure-stability/outputs/physics_feature_analysis_v2/class_analysis_features.csv
TRAIN_STRUCTURE_CSV: /media/hdd0/whyz/structure-stability/data/train_structure.csv
DEV_STRUCTURE_CSV: /media/hdd0/whyz/structure-stability/data/dev_structure.csv
device: cuda


In [2]:
train_df = pd.read_csv(DATA_DIR / 'train.csv', encoding='utf-8-sig')
dev_df = pd.read_csv(DATA_DIR / 'dev.csv', encoding='utf-8-sig')
test_df = pd.read_csv(DATA_DIR / 'sample_submission.csv', encoding='utf-8-sig')
feature_df = pd.read_csv(FEATURE_CSV, encoding='utf-8-sig')
train_structure_df = pd.read_csv(TRAIN_STRUCTURE_CSV, encoding='utf-8-sig')
dev_structure_df = pd.read_csv(DEV_STRUCTURE_CSV, encoding='utf-8-sig')

feature_df = feature_df[['split', 'sample_id'] + ALL_TEACHER_FEATURES].copy()
feature_df = feature_df.rename(columns={'sample_id': 'id'})

for split_name, split_df in [('train', train_df), ('dev', dev_df)]:
    part = feature_df[feature_df['split'].eq(split_name)].drop(columns=['split']).copy()
    missing = set(split_df['id']) - set(part['id'])
    print(split_name, 'teacher rows:', len(part), 'missing:', len(missing))

physics_scaler = StandardScaler()
image_scaler = StandardScaler()
train_phys_mask = feature_df['split'].eq('train') & feature_df[PHYSICS_FEATURES].notna().all(axis=1)
train_img_mask = feature_df['split'].eq('train') & feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
physics_scaler.fit(feature_df.loc[train_phys_mask, PHYSICS_FEATURES])
image_scaler.fit(feature_df.loc[train_img_mask, IMAGE_STRUCTURE_FEATURES])

all_phys_mask = feature_df[PHYSICS_FEATURES].notna().all(axis=1)
all_img_mask = feature_df[IMAGE_STRUCTURE_FEATURES].notna().all(axis=1)
feature_df.loc[all_phys_mask, PHYSICS_FEATURES] = physics_scaler.transform(feature_df.loc[all_phys_mask, PHYSICS_FEATURES])
feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES] = image_scaler.transform(feature_df.loc[all_img_mask, IMAGE_STRUCTURE_FEATURES])

print('teacher target normalization: StandardScaler fitted on train split')
print('physics normalized rows:', int(all_phys_mask.sum()), 'image normalized rows:', int(all_img_mask.sum()))

train_df = train_df.merge(feature_df[feature_df['split'].eq('train')].drop(columns=['split']), on='id', how='left')
dev_df = dev_df.merge(feature_df[feature_df['split'].eq('dev')].drop(columns=['split']), on='id', how='left')

train_structure_df = train_structure_df[['id', STRUCTURE_INDEX_COL]].copy()
dev_structure_df = dev_structure_df[['id', STRUCTURE_INDEX_COL]].copy()

all_structure = pd.concat([train_structure_df[[STRUCTURE_INDEX_COL]], dev_structure_df[[STRUCTURE_INDEX_COL]]], ignore_index=True)
structure_values = sorted(int(x) for x in all_structure[STRUCTURE_INDEX_COL].dropna().unique().tolist())
structure_to_class = {v: i for i, v in enumerate(structure_values)}
NUM_STRUCTURE_CLASSES = len(structure_values)
assert NUM_STRUCTURE_CLASSES > 1, 'structure classes must be >= 2'
print('NUM_STRUCTURE_CLASSES:', NUM_STRUCTURE_CLASSES)

train_structure_df[STRUCTURE_CLASS_COL] = train_structure_df[STRUCTURE_INDEX_COL].map(structure_to_class)
dev_structure_df[STRUCTURE_CLASS_COL] = dev_structure_df[STRUCTURE_INDEX_COL].map(structure_to_class)

train_df = train_df.merge(train_structure_df[['id', STRUCTURE_CLASS_COL]], on='id', how='left')
dev_df = dev_df.merge(dev_structure_df[['id', STRUCTURE_CLASS_COL]], on='id', how='left')

print('train structure labeled rows:', int(train_df[STRUCTURE_CLASS_COL].notna().sum()))
print('dev structure labeled rows:', int(dev_df[STRUCTURE_CLASS_COL].notna().sum()))

test_df['sample_dir'] = str(DATA_DIR / 'test')

print('train:', train_df.shape)
print('dev:', dev_df.shape)



train teacher rows: 1000 missing: 0
dev teacher rows: 100 missing: 0
teacher target normalization: StandardScaler fitted on train split
physics normalized rows: 1100 image normalized rows: 1100
NUM_STRUCTURE_CLASSES: 7
train structure labeled rows: 1000
dev structure labeled rows: 100
train: (1000, 13)
dev: (100, 13)


In [3]:

def _extract_frame_by_sec(cap, sec, fps, frame_count):
    frame_idx = int(max(0, min(frame_count - 1, round(sec * fps))))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _extract_last_frame(cap, frame_count):
    last_idx = max(0, frame_count - 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, last_idx)
    ok, frame = cap.read()
    if not ok:
        return None
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def _video_aug_cache_signature(cfg):
    keys = [
        'SEED',
        'UNSTABLE_START_MIN_SEC',
        'UNSTABLE_START_MAX_SEC',
        'UNSTABLE_FRAMES_MIN',
        'UNSTABLE_FRAMES_MAX',
        'STABLE_END_MIN_SEC',
        'STABLE_END_MAX_SEC',
        'STABLE_FRAMES_PER_VIDEO',
    ]
    return {k: cfg.get(k) for k in keys}


def build_video_augmented_df(train_df, data_dir, cfg):
    train_root = data_dir / 'train'
    aug_root = data_dir / 'train_video_aug'
    aug_root.mkdir(parents=True, exist_ok=True)

    cache_csv = aug_root / 'aug_df.csv'
    cache_meta = aug_root / 'cache_meta.json'
    cache_sig = _video_aug_cache_signature(cfg)

    if cfg.get('VIDEO_AUG_CACHE', True) and cache_csv.exists() and cache_meta.exists():
        try:
            meta = json.loads(cache_meta.read_text())
            if meta.get('signature') == cache_sig:
                cached_df = pd.read_csv(cache_csv)
                if not cached_df.empty:
                    cached_df['sample_dir'] = str(aug_root)
                    return cached_df
        except Exception as exc:
            print('video aug cache read failed:', exc)

    for p in aug_root.glob('AUGV_*'):
        if p.is_dir():
            shutil.rmtree(p, ignore_errors=True)

    rng = np.random.default_rng(cfg['SEED'])
    saved_idx = 0
    aug_rows = []

    def save_aug(img, label):
        nonlocal saved_idx
        aug_id = f'AUGV_{saved_idx:07d}'
        out_dir = aug_root / aug_id
        out_dir.mkdir(parents=True, exist_ok=True)
        Image.fromarray(img).save(out_dir / 'front.png')
        Image.fromarray(img).save(out_dir / 'top.png')
        row = {'id': aug_id, 'label': label, 'sample_dir': str(aug_root)}
        for col in ALL_TEACHER_FEATURES:
            row[col] = np.nan
        row[STRUCTURE_CLASS_COL] = np.nan
        aug_rows.append(row)
        saved_idx += 1

    for row in tqdm(train_df.itertuples(index=False), total=len(train_df), desc='video aug', dynamic_ncols=True, ascii=True):
        sample_id = row.id
        label = row.label
        video_path = train_root / sample_id / 'simulation.mp4'
        if not video_path.exists():
            continue

        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            continue

        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if fps <= 0 or frame_count <= 0:
            cap.release()
            continue

        if label == 'unstable':
            n_frames = int(rng.integers(cfg['UNSTABLE_FRAMES_MIN'], cfg['UNSTABLE_FRAMES_MAX'] + 1))
            secs = rng.uniform(cfg['UNSTABLE_START_MIN_SEC'], cfg['UNSTABLE_START_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)
        else:
            n_frames = int(cfg['STABLE_FRAMES_PER_VIDEO'])
            secs = rng.uniform(cfg['STABLE_END_MIN_SEC'], cfg['STABLE_END_MAX_SEC'], size=n_frames)
            for sec in secs:
                frame = _extract_frame_by_sec(cap, float(sec), fps, frame_count)
                if frame is not None:
                    save_aug(frame, label)

        cap.release()

    aug_df = pd.DataFrame(aug_rows)
    aug_df.to_csv(cache_csv, index=False)
    cache_meta.write_text(json.dumps({'signature': cache_sig}, indent=2))
    return aug_df


class TeacherRegularizationDataset(Dataset):
    def __init__(self, df, root_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform
        self.is_test = is_test
        self.label_map = {'stable': 0, 'unstable': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = str(row['id'])
        base_dir = self.root_dir
        if ('sample_dir' in self.df.columns) and pd.notna(row.get('sample_dir', np.nan)):
            base_dir = str(row['sample_dir'])
        folder_path = os.path.join(base_dir, sample_id)

        views = []
        for name in ['front', 'top']:
            img_path = os.path.join(folder_path, f'{name}.png')
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            views.append(image)

        if self.is_test:
            return {'views': views, 'id': sample_id}

        label = self.label_map[row['label']]
        physics_target = row[PHYSICS_FEATURES].to_numpy(dtype=np.float32)
        image_target = row[IMAGE_STRUCTURE_FEATURES].to_numpy(dtype=np.float32)
        structure_target = row.get(STRUCTURE_CLASS_COL, np.nan)
        structure_target = int(structure_target) if pd.notna(structure_target) else -1
        return {
            'views': views,
            'label': torch.tensor(label, dtype=torch.float32),
            'physics_target': torch.tensor(physics_target, dtype=torch.float32),
            'image_target': torch.tensor(image_target, dtype=torch.float32),
            'structure_target': torch.tensor(structure_target, dtype=torch.long),
            'has_teacher': torch.tensor(float(np.isfinite(physics_target).all() and np.isfinite(image_target).all()), dtype=torch.float32),
            'id': sample_id,
        }


train_transform, test_transform = build_default_transforms(CFG['IMG_SIZE'])
train_df_for_train = train_df.copy()
train_df_for_train['sample_dir'] = str(DATA_DIR / 'train')
if CFG['VIDEO_AUG_ENABLE']:
    aug_df = build_video_augmented_df(train_df, DATA_DIR, CFG)
    if not aug_df.empty:
        train_df_for_train = pd.concat([train_df_for_train, aug_df], ignore_index=True)

dev_domain_df = dev_df.copy()
dev_domain_df['sample_dir'] = str(DATA_DIR / 'dev')
dev_eval_df = dev_df.copy()
dev_eval_df['sample_dir'] = str(DATA_DIR / 'dev')

train_dataset = TeacherRegularizationDataset(train_df_for_train, str(DATA_DIR / 'train'), train_transform, is_test=False)
dev_domain_dataset = TeacherRegularizationDataset(dev_domain_df, str(DATA_DIR / 'dev'), train_transform, is_test=False)
dev_eval_dataset = TeacherRegularizationDataset(dev_eval_df, str(DATA_DIR / 'dev'), test_transform, is_test=False)
test_dataset = TeacherRegularizationDataset(test_df, str(DATA_DIR / 'test'), test_transform, is_test=True)

loader_kwargs = dict(
    batch_size=CFG['BATCH_SIZE'],
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=(device.type == 'cuda'),
)
train_loader = DataLoader(train_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED']), **loader_kwargs)
dev_domain_loader = DataLoader(dev_domain_dataset, shuffle=True, worker_init_fn=seed_worker, generator=make_generator(CFG['SEED'] + 1), **loader_kwargs)
dev_eval_loader = DataLoader(dev_eval_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

print('train_dataset:', len(train_dataset))
print('dev_domain_dataset:', len(dev_domain_dataset))
print('dev_eval_dataset:', len(dev_eval_dataset))
print('test_dataset:', len(test_dataset))



train_dataset: 3000
dev_domain_dataset: 100
dev_eval_dataset: 100
test_dataset: 1000


In [4]:

@dataclass(frozen=True)
class TeacherRegularizationConfig:
    backbone_name: str = CFG['BACKBONE_NAME']
    pretrained: bool = True
    attn_dim: int = CFG['ATTN_DIM']
    num_heads: int = CFG['NUM_HEADS']
    num_layers: int = CFG['NUM_LAYERS']
    pos_grid: int = CFG['POS_GRID']
    dropout: float = CFG['DROPOUT']
    classifier_hidden_dim: int = CFG['CLASSIFIER_HIDDEN_DIM']
    classifier_mid_dim: int = CFG['CLASSIFIER_MID_DIM']
    classifier_dropout: float = CFG['CLASSIFIER_DROPOUT']
    domain_hidden_dim: int = CFG['DOMAIN_HIDDEN_DIM']
    domain_dropout: float = CFG['DOMAIN_DROPOUT']
    physics_dim: int = len(PHYSICS_FEATURES)
    image_dim: int = len(IMAGE_STRUCTURE_FEATURES)
    structure_num_classes: int = NUM_STRUCTURE_CLASSES
    out_dim: int = 1


class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_ * grad_output, None


def grad_reverse(x, lambda_=1.0):
    return GradientReversalFunction.apply(x, lambda_)


class CrossAttentionBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        self.norm_q = nn.LayerNorm(dim)
        self.norm_kv = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)

    def forward(self, q_tokens, kv_tokens):
        q = self.norm_q(q_tokens)
        kv = self.norm_kv(kv_tokens)
        attn_out, _ = self.attn(q, kv, kv, need_weights=False)
        return q_tokens + attn_out


class TeacherRegularizedMultiView(nn.Module):
    def __init__(self, config: TeacherRegularizationConfig | None = None):
        super().__init__()
        self.config = config or TeacherRegularizationConfig()
        self.backbone = timm.create_model(
            self.config.backbone_name,
            pretrained=self.config.pretrained,
            num_classes=0,
            global_pool='',
        )
        feature_dim = self.backbone.num_features
        self.proj = nn.Conv2d(feature_dim, self.config.attn_dim, kernel_size=1, bias=False)
        self.pos_embed = nn.Parameter(torch.randn(1, self.config.attn_dim, self.config.pos_grid, self.config.pos_grid) * 0.02)
        self.cross_12 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.cross_21 = nn.ModuleList([CrossAttentionBlock(self.config.attn_dim, self.config.num_heads, self.config.dropout) for _ in range(self.config.num_layers)])
        self.classifier = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim),
            nn.BatchNorm1d(self.config.classifier_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.classifier_dropout),
            nn.Linear(self.config.classifier_hidden_dim, self.config.classifier_mid_dim),
            nn.ReLU(),
            nn.Linear(self.config.classifier_mid_dim, self.config.out_dim),
        )
        self.physics_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.physics_dim),
        )
        self.image_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.image_dim),
        )
        self.structure_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.classifier_hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(self.config.classifier_hidden_dim // 2, self.config.structure_num_classes),
        )
        self.domain_head = nn.Sequential(
            nn.Linear(self.config.attn_dim * 2, self.config.domain_hidden_dim),
            nn.ReLU(),
            nn.Dropout(self.config.domain_dropout),
            nn.Linear(self.config.domain_hidden_dim, 1),
        )

    def _to_tokens(self, feat_map):
        feat_map = self.proj(feat_map)
        pos = F.interpolate(self.pos_embed, size=feat_map.shape[-2:], mode='bilinear', align_corners=False)
        feat_map = feat_map + pos
        return feat_map.flatten(2).transpose(1, 2)

    def extract_features(self, views):
        f1 = self.backbone.forward_features(views[0])
        f2 = self.backbone.forward_features(views[1])
        t1 = self._to_tokens(f1)
        t2 = self._to_tokens(f2)
        for blk12, blk21 in zip(self.cross_12, self.cross_21):
            t1_prev, t2_prev = t1, t2
            t1 = blk12(t1_prev, t2_prev)
            t2 = blk21(t2_prev, t1_prev)
        return torch.cat([t1.mean(dim=1), t2.mean(dim=1)], dim=1)

    def forward(self, views, lambda_=0.0):
        feat = self.extract_features(views)
        out = {
            'class_logit': self.classifier(feat).view(-1),
            'physics_pred': self.physics_head(feat),
            'image_pred': self.image_head(feat),
            'feat': feat,
            'structure_logit': self.structure_head(feat),
        }
        dom_feat = grad_reverse(feat, lambda_)
        out['domain_logit'] = self.domain_head(dom_feat).view(-1)
        return out



In [5]:

def mixup_multiview_batch(views, labels, alpha=0.2):
    if alpha <= 0:
        return views, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = labels.size(0)
    index = torch.randperm(batch_size, device=labels.device)
    mixed_views = [lam * v + (1.0 - lam) * v[index, :] for v in views]
    return mixed_views, labels, labels[index], lam


def compute_lambda(epoch_idx, step_idx, steps_per_epoch, total_epochs, warmup_epochs=0, max_lambda=1.0, gamma=10.0):
    total_steps = max(1, total_epochs * steps_per_epoch)
    current_step = max(0, (epoch_idx - 1) * steps_per_epoch + step_idx)
    progress = current_step / total_steps
    warmup_progress = warmup_epochs / max(1, total_epochs)
    if progress <= warmup_progress:
        return 0.0
    effective_progress = (progress - warmup_progress) / max(1e-8, 1.0 - warmup_progress)
    lambda_base = 2.0 / (1.0 + np.exp(-gamma * effective_progress)) - 1.0
    return float(max_lambda * lambda_base)


def masked_smooth_l1(pred, target):
    mask = torch.isfinite(target).all(dim=1)
    if mask.any():
        return F.smooth_l1_loss(pred[mask], target[mask])
    return pred.sum() * 0.0


def masked_cross_entropy(logits, target, ignore_index=-1):
    mask = target.ne(ignore_index)
    if mask.any():
        return F.cross_entropy(logits[mask], target[mask])
    return logits.sum() * 0.0


def train_one_epoch(model, train_loader, dev_loader, optimizer, device, epoch_idx, total_epochs, ema=None):
    model.train()
    dev_iter = cycle(dev_loader)
    total_rows = []
    steps = len(train_loader)
    for step_idx, batch in enumerate(tqdm(train_loader, desc='Training', dynamic_ncols=True, ascii=True), start=1):
        dev_batch = next(dev_iter)
        train_views = [v.to(device) for v in batch['views']]
        train_labels = batch['label'].to(device).float()
        train_phys = batch['physics_target'].to(device)
        train_img = batch['image_target'].to(device)
        train_structure = batch['structure_target'].to(device)
        dev_views = [v.to(device) for v in dev_batch['views']]

        optimizer.zero_grad()

        if CFG['MIXUP_ALPHA'] > 0 and np.random.rand() < CFG['MIXUP_PROB']:
            mixed_views, labels_a, labels_b, lam = mixup_multiview_batch(train_views, train_labels, CFG['MIXUP_ALPHA'])
            outputs = model(mixed_views, lambda_=0.0)
            loss_cls = lam * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_a) + (1.0 - lam) * F.binary_cross_entropy_with_logits(outputs['class_logit'], labels_b)
            loss_phys = outputs['physics_pred'].sum() * 0.0
            loss_img = outputs['image_pred'].sum() * 0.0
            loss_struct = outputs['structure_logit'].sum() * 0.0
        else:
            outputs = model(train_views, lambda_=0.0)
            loss_cls = F.binary_cross_entropy_with_logits(outputs['class_logit'], train_labels)
            loss_phys = masked_smooth_l1(outputs['physics_pred'], train_phys)
            loss_img = masked_smooth_l1(outputs['image_pred'], train_img)
            loss_struct = masked_cross_entropy(outputs['structure_logit'], train_structure)

        domain_views = [torch.cat([tv, dv], dim=0) for tv, dv in zip(train_views, dev_views)]
        domain_labels = torch.cat([
            torch.zeros(train_views[0].size(0), device=device),
            torch.ones(dev_views[0].size(0), device=device),
        ], dim=0)
        grl_lambda = compute_lambda(
            epoch_idx,
            step_idx - 1,
            steps,
            total_epochs,
            warmup_epochs=CFG['GRL_WARMUP_EPOCHS'],
            max_lambda=CFG['GRL_MAX_LAMBDA'],
            gamma=CFG['GRL_GAMMA'],
        )
        domain_outputs = model(domain_views, lambda_=grl_lambda)
        loss_dom = F.binary_cross_entropy_with_logits(domain_outputs['domain_logit'], domain_labels)

        loss = (
            loss_cls
            + CFG['PHYSICS_LOSS_WEIGHT'] * loss_phys
            + CFG['IMAGE_LOSS_WEIGHT'] * loss_img
            + CFG['STRUCTURE_LOSS_WEIGHT'] * loss_struct
            + CFG['DOMAIN_LOSS_WEIGHT'] * loss_dom
        )
        loss.backward()
        optimizer.step()
        if ema is not None:
            ema.update(model)

        with torch.no_grad():
            domain_probs = torch.sigmoid(domain_outputs['domain_logit'])
            domain_acc = ((domain_probs > 0.5) == domain_labels).float().mean().item()

        total_rows.append({
            'loss': float(loss.item()),
            'loss_cls': float(loss_cls.item()),
            'loss_phys': float(loss_phys.item()),
            'loss_img': float(loss_img.item()),
            'loss_struct': float(loss_struct.item()),
            'loss_dom': float(loss_dom.item()),
            'domain_acc': float(domain_acc),
        })

    hist = pd.DataFrame(total_rows)
    return hist.mean().to_dict()


@torch.no_grad()
def evaluate_classification(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []
    physics_losses = []
    image_losses = []
    structure_losses = []
    for batch in tqdm(loader, desc='Dev Eval', dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in batch['views']]
        labels = batch['label'].to(device).float()
        physics_target = batch['physics_target'].to(device)
        image_target = batch['image_target'].to(device)
        structure_target = batch['structure_target'].to(device)
        outputs = model(views, lambda_=0.0)
        probs = torch.sigmoid(outputs['class_logit'])
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        physics_losses.append(float(masked_smooth_l1(outputs['physics_pred'], physics_target).item()))
        image_losses.append(float(masked_smooth_l1(outputs['image_pred'], image_target).item()))
        structure_losses.append(float(masked_cross_entropy(outputs['structure_logit'], structure_target).item()))

    all_probs = np.array(all_probs, dtype=np.float64)
    all_labels = np.array(all_labels, dtype=np.float64)
    p = np.clip(all_probs, 1e-15, 1 - 1e-15)
    logloss = float(-np.mean(all_labels * np.log(p) + (1.0 - all_labels) * np.log(1.0 - p)))
    acc = float(np.mean((all_probs > 0.5) == all_labels))
    auc = float(__import__('sklearn.metrics').metrics.roc_auc_score(all_labels, all_probs))
    return {
        'dev_logloss': logloss,
        'dev_acc': acc,
        'dev_auc': auc,
        'dev_phys_loss': float(np.mean(physics_losses)),
        'dev_img_loss': float(np.mean(image_losses)),
        'dev_structure_loss': float(np.mean(structure_losses)),
    }


@torch.no_grad()
def predict_test_probs(model, loader, device):
    model.eval()
    probs_all = []
    ids = []
    for batch in tqdm(loader, desc='Test Inference', dynamic_ncols=True, ascii=True):
        views = [v.to(device) for v in batch['views']]
        outputs = model(views, lambda_=0.0)
        probs = torch.sigmoid(outputs['class_logit'])
        probs_all.extend(probs.cpu().numpy())
        ids.extend(batch['id'])
    return ids, np.array(probs_all, dtype=np.float64)



In [6]:

model = TeacherRegularizedMultiView(TeacherRegularizationConfig()).to(device)
optimizer = optim.Adam(model.parameters(), lr=CFG['LEARNING_RATE'], weight_decay=CFG['WEIGHT_DECAY'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['EPOCHS'], eta_min=CFG['MIN_LR'])
ema = ModelEMA(model, EMAConfig(decay=CFG['EMA_DECAY']))

artifacts = allocate_output_paths(experiment_name='teacher_regularization', major_version='v4')
best_model_path = artifacts['weight_path']
submission_path = artifacts['submission_path']
history_path = (Path.cwd() / '../../../outputs/eda_preprocessing/teacher_regularization_v4.0_history.csv').resolve()
history_path.parent.mkdir(parents=True, exist_ok=True)
print('Artifact version:', artifacts['version'])
print('best_model_path:', best_model_path)
print('submission_path:', submission_path)

best_logloss = float('inf')
best_epoch = -1
patience_left = CFG['EARLY_STOPPING_PATIENCE']
history = []

for epoch in range(1, CFG['EPOCHS'] + 1):
    train_metrics = train_one_epoch(model, train_loader, dev_domain_loader, optimizer, device, epoch, CFG['EPOCHS'], ema=ema)
    eval_model = ema.ema_model if CFG['EMA_USE_FOR_EVAL'] else model
    dev_metrics = evaluate_classification(eval_model, dev_eval_loader, device)

    improved = dev_metrics['dev_logloss'] < best_logloss
    if improved:
        best_logloss = dev_metrics['dev_logloss']
        best_epoch = epoch
        patience_left = CFG['EARLY_STOPPING_PATIENCE']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'ema_state_dict': ema.ema_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'cfg': CFG,
            **dev_metrics,
        }, best_model_path)
    else:
        patience_left -= 1

    scheduler.step()
    row = {
        'epoch': epoch,
        **{k: float(v) for k, v in train_metrics.items()},
        **dev_metrics,
        'lr': float(optimizer.param_groups[0]['lr']),
        'improved': bool(improved),
        'patience_left': int(patience_left),
    }
    history.append(row)
    print(row)
    if patience_left < 0:
        print('early stop:', epoch)
        break

history_df = pd.DataFrame(history)
history_df.to_csv(history_path, index=False)
print('saved history:', history_path)



Artifact version: v4.0
best_model_path: /media/hdd0/whyz/structure-stability/outputs/weights/teacher_regularization_v4.0.pt
submission_path: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v4.0.csv


Dev Eval: 100%|##########| 7/7 [00:01<00:00,  6.35it/s]


{'epoch': 1, 'loss': 0.48948854192140256, 'loss_cls': 0.26920864003849154, 'loss_phys': 0.23402116450659455, 'loss_img': 0.2540883578984265, 'loss_struct': 0.49188938728958476, 'loss_dom': 0.4893726356961626, 'domain_acc': 0.7588763322601927, 'dev_logloss': 0.6512728124742138, 'dev_acc': 0.66, 'dev_auc': 0.890224358974359, 'dev_phys_loss': 2.076498900141035, 'dev_img_loss': 0.7253956965037754, 'dev_structure_loss': 1.5397111858640398, 'lr': 0.0002999262307746768, 'improved': True, 'patience_left': 5}


Dev Eval: 100%|##########| 7/7 [00:00<00:00,  9.35it/s]


{'epoch': 2, 'loss': 0.30883230728671907, 'loss_cls': 0.16028787743577616, 'loss_phys': 0.1656351940337311, 'loss_img': 0.17745767840793594, 'loss_struct': 0.22843622728365479, 'loss_dom': 0.3711843646745733, 'domain_acc': 0.8323138308651904, 'dev_logloss': 0.5426908269471986, 'dev_acc': 0.88, 'dev_auc': 0.9859775641025641, 'dev_phys_loss': 2.068238275391715, 'dev_img_loss': 0.7094279612813678, 'dev_structure_loss': 0.9555733374186924, 'lr': 0.00029970499590002654, 'improved': True, 'patience_left': 5}


...

{'epoch': 21, 'loss': 0.1548891310441367, 'loss_cls': 0.03908241371731918, 'loss_phys': 0.05011161170655891, 'loss_img': 0.04801603693061786, 'loss_struct': 0.08234868095128826, 'loss_dom': 0.4642634975149276, 'domain_acc': 0.7638297879949529, 'dev_logloss': 0.039772138998872386, 'dev_acc': 0.98, 'dev_auc': 1.0, 'dev_phys_loss': 2.0824755089623586, 'dev_img_loss': 0.5657273530960083, 'dev_structure_loss': 0.05218953000647681, 'lr': 0.0002686281743501657, 'improved': False, 'patience_left': 1}



Dev Eval: 100%|##########| 7/7 [00:00<00:00,  9.22it/s]

{'epoch': 22, 'loss': 0.1555720088171198, 'loss_cls': 0.03684538146115572, 'loss_phys': 0.05553487727933742, 'loss_img': 0.03879626213784696, 'loss_struct': 0.08268782477432282, 'loss_dom': 0.4815408637390492, 'domain_acc': 0.7743461899300839, 'dev_logloss': 0.045505929101137335, 'dev_acc': 0.98, 'dev_auc': 1.0, 'dev_phys_loss': 2.0813990150179182, 'dev_img_loss': 0.5642056379999433, 'dev_structure_loss': 0.04613485799304077, 'lr': 0.0002656917297949805, 'improved': False, 'patience_left': 0}



Dev Eval: 100%|##########| 7/7 [00:00<00:00,  9.15it/s]

{'epoch': 23, 'loss': 0.21958720941651375, 'loss_cls': 0.06224135590216858, 'loss_phys': 0.06328153545705681, 'loss_img': 0.048292572603268705, 'loss_struct': 0.10825045738137634, 'loss_dom': 0.6489234450966754, 'domain_acc': 0.7106382991088197, 'dev_logloss': 0.05103866749887248, 'dev_acc': 0.97, 'dev_auc': 1.0, 'dev_phys_loss': 2.0787589379719327, 'dev_img_loss': 0.5625300662858146, 'dev_structure_loss': 0.03881872019597462, 'lr': 0.0002626416049097537, 'improved': False, 'patience_left': -1}
early stop: 23
saved history: /media/hdd0/whyz/outputs/eda_preprocessing/teacher_regularization_v4.0_history.csv


In [7]:

checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
best_state = checkpoint['ema_state_dict'] if CFG['EMA_USE_FOR_EVAL'] else checkpoint['model_state_dict']
model.load_state_dict(best_state)
final_dev_metrics = evaluate_classification(model, dev_eval_loader, device)
print('best_epoch:', best_epoch)
print(final_dev_metrics)

ids, test_probs = predict_test_probs(model, test_loader, device)
submission = pd.DataFrame({
    'id': ids,
    'unstable_prob': test_probs,
    'stable_prob': 1.0 - test_probs,
})
submission.to_csv(submission_path, index=False, encoding='utf-8-sig')
print('saved submission:', submission_path)
submission.head()


Dev Eval: 100%|##########| 7/7 [00:00<00:00,  8.50it/s]


best_epoch: 17
{'dev_logloss': 0.016546577683531863, 'dev_acc': 1.0, 'dev_auc': 1.0, 'dev_phys_loss': 2.0865075247628346, 'dev_img_loss': 0.5709131913525718, 'dev_structure_loss': 0.049843624234199524}


Test Inference: 100%|##########| 63/63 [00:03<00:00, 15.94it/s]

saved submission: /media/hdd0/whyz/structure-stability/outputs/submissions/teacher_regularization_v4.0.csv


,id,unstable_prob,stable_prob
0,TEST_0001,0.004470,0.995530
1,TEST_0002,0.999046,0.000954
2,TEST_0003,0.999954,0.000046
3,TEST_0004,0.998622,0.001378
4,TEST_0005,0.000431,0.999569
